# The Gilded Age: historical evidence and economic mechanisms

This notebook distinguishes historical observations, literature-derived average rates, and hypothetical economic models. It does not estimate historical treatment effects. The manuscript and README explain the provenance and scope of each series.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import numpy as np
from analysis import (LinearMarket, average_cost, compounded_index, foreign_born_data,
                      generate_all, growth_data, labor_share_path, market_outcome,
                      railroad_data, wealth_data)


## Source-derived series
Each source is transcribed into a small CSV. There are no silently invented observations for missing years.

In [ ]:
rail_years, rail_miles = railroad_data()
print('Railroad growth factor, 1870–1900:', round(float(rail_miles[-1] / rail_miles[0]), 3))
foreign_years, foreign_count, foreign_share = foreign_born_data()
print('Foreign-born count and percentage in 1890:', float(foreign_count[4]), float(foreign_share[4]))


In [ ]:
wealth_years, wealth_shares = wealth_data()
assert 1890 not in wealth_years
print('Massachusetts top-one-percent taxable wealth share in 1900:',
      float(wealth_shares[np.where(wealth_years == 1900)[0][0], 2]))
for label, growth, years in growth_data():
    print(label, round(compounded_index(growth, years), 1), 'constructed endpoint')


## Mechanisms, not historical measurements
The following parameter choices are arbitrary demonstration values, not nineteenth-century prices, costs, or labor shares.

In [ ]:
example = LinearMarket(intercept=12.0, slope=0.5, cost=2.0)
result = market_outcome(example)
print('Competitive and monopoly quantities:', result.competitive_quantity, result.monopoly_quantity)
print('Deadweight loss in toy model:', result.deadweight_loss)
print('Average costs at different scales:', average_cost(np.array([2., 10., 30.]), 40., 2.))


In [ ]:
years, rising = labor_share_path(0.025, 0.60, 0.66)
_, constant = labor_share_path(0.025, 0.60, 0.60)
_, falling = labor_share_path(0.025, 0.60, 0.48)
assert rising[-1] > constant[-1] > falling[-1]
print('Three hypothetical final wage indexes:', round(rising[-1], 1),
      round(constant[-1], 1), round(falling[-1], 1))


## End-to-end reproducibility
Generate all figures and derived tables in a temporary directory without touching checked-in artifacts.

In [ ]:
with TemporaryDirectory() as directory:
    generate_all(Path(directory))
    assert len(list((Path(directory) / 'figures').glob('*.png'))) == 8
    assert len(list((Path(directory) / 'results').glob('*.csv'))) == 3
